In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [2]:
# Load and combine
df_2023 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2023_c20260323.csv')
df_2024 = pd.read_csv('../data/StormEvents_details-ftp_v1.0_d2024_c20260323 2.csv')
df = pd.concat([df_2023, df_2024], ignore_index=True)

# Filter to top 10 event types
top_10_types = df['EVENT_TYPE'].value_counts().head(10).index.tolist()
print(df['EVENT_TYPE'].value_counts().head(10))
df = df[df['EVENT_TYPE'].isin(top_10_types)].copy()

# Feature engineering (all vectorized, no loops)
month_map = {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
             'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
df['MONTH'] = df['MONTH_NAME'].map(month_map)

# Hour from BEGIN_TIME (stored as HHMM integer, e.g. 1430 -> 14)
df['HOUR'] = df['BEGIN_TIME'] // 100

# Duration in minutes from full timestamps
df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'], format='%d-%b-%y %H:%M:%S')
df['DURATION_MIN'] = (df['END_DT'] - df['BEGIN_DT']).dt.total_seconds() / 60

# CZ_TYPE to binary
df['CZ_TYPE_ENC'] = (df['CZ_TYPE'] == 'C').astype(int)

# Label encode state
le_state = LabelEncoder()
df['STATE_ENC'] = le_state.fit_transform(df['STATE'].astype(str))

# Label encode WFO
le_wfo = LabelEncoder()
df['WFO_ENC'] = le_wfo.fit_transform(df['WFO'].astype(str))

# Encode target labels (XGBoost wants integer classes)
le_y = LabelEncoder()
y_enc = le_y.fit_transform(df['EVENT_TYPE'])

# Stratified split (before imputation to prevent data leakage)
feature_cols = ['MONTH', 'HOUR', 'STATE_ENC', 'BEGIN_LAT', 'BEGIN_LON',
                'DURATION_MIN', 'CZ_TYPE_ENC', 'WFO_ENC']
X = df[feature_cols]
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, stratify=y_enc, test_size=0.2, random_state=42
)

# --- Hierarchical lat/lon imputation (fit on training data only) ---
# Compute median coords at three geographic levels from training rows that have coords
train_df = df.loc[X_train.index]
has_coords = train_df[train_df['BEGIN_LAT'].notna()]
median_cz = has_coords.groupby(['STATE', 'CZ_FIPS'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_wfo = has_coords.groupby(['STATE', 'WFO'])[['BEGIN_LAT', 'BEGIN_LON']].median()
median_state = has_coords.groupby('STATE')[['BEGIN_LAT', 'BEGIN_LON']].median()

# Apply hierarchical fill to full df, then rebuild X
df_imp = df.copy()
missing = df_imp['BEGIN_LAT'].isna()

# Level 1: (STATE, CZ_FIPS) — county/zone median
for (state, cz_fips), row in median_cz.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['CZ_FIPS'] == cz_fips)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 2: (STATE, WFO) — forecast office region median
for (state, wfo), row in median_wfo.iterrows():
    mask = missing & (df_imp['STATE'] == state) & (df_imp['WFO'] == wfo)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

missing = df_imp['BEGIN_LAT'].isna()

# Level 3: STATE — state median (last resort)
for state, row in median_state.iterrows():
    mask = missing & (df_imp['STATE'] == state)
    df_imp.loc[mask, ['BEGIN_LAT', 'BEGIN_LON']] = row.values

# Rebuild X from imputed data using the same indices
X_imp = df_imp[feature_cols]
X_train = X_imp.loc[X_train.index]
X_test = X_imp.loc[X_test.index]

# Sanity checks
print(f"\nShape: {X_train.shape}")
print(f"Missing per column (train):\n{X_train.isna().sum()}")
print(f"Missing per column (test):\n{X_test.isna().sum()}")

EVENT_TYPE
Thunderstorm Wind           41177
Hail                        20702
Drought                      9288
Flash Flood                  8286
Excessive Heat               7684
High Wind                    7485
Heat                         6494
Winter Weather               6448
Flood                        5318
Marine Thunderstorm Wind     4835
Name: count, dtype: int64

Shape: (94173, 8)
Missing per column (train):
MONTH           0
HOUR            0
STATE_ENC       0
BEGIN_LAT       0
BEGIN_LON       0
DURATION_MIN    0
CZ_TYPE_ENC     0
WFO_ENC         0
dtype: int64
Missing per column (test):
MONTH           0
HOUR            0
STATE_ENC       0
BEGIN_LAT       0
BEGIN_LON       0
DURATION_MIN    0
CZ_TYPE_ENC     0
WFO_ENC         0
dtype: int64


In [3]:
# from sklearn.model_selection import GridSearchCV, StratifiedKFold
#
# # Hyperparameter grid (proposal: learning rate, max depth, n_estimators, subsample)
# param_grid = {
#     'learning_rate': [0.01, 0.05, 0.1, 0.3],
#     'max_depth': [3, 5, 7, 9],
#     'n_estimators': [100, 300, 500, 700],
#     'subsample': [0.7, 0.8, 1.0],
# }
#
# base_model = xgb.XGBClassifier(
#     objective='multi:softmax',
#     num_class=10,
#     random_state=42,
#     tree_method='hist',
# )
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#
# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     scoring='f1_macro',
#     cv=cv,
#     verbose=1,
#     n_jobs=-1,
# )
#
# grid_search.fit(X_train, y_train)
#
# print(f"\nBest CV Macro F1: {grid_search.best_score_:.4f}")
# print(f"Best params: {grid_search.best_params_}")
#
# # Use the best model for evaluation
# #model = grid_search.best_estimator_

# Best hyperparameters from the Week 2 grid search (5-fold stratified CV, 192 candidates).
# Skipping re-run — those results are already logged in the Week 2 report.
best_params = {
    'learning_rate': 0.1,
    'max_depth': 9,
    'n_estimators': 500,
    'subsample': 0.7,
}
best_cv_macro_f1 = 0.8949  # from the Week 2 grid search

print(f"Using tuned XGBoost params: {best_params}")
print(f"Prior CV Macro F1: {best_cv_macro_f1:.4f}")

# Train the tuned model with the best params (used by the rest of the notebook)
model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
    **best_params,
)
model.fit(X_train, y_train)

Using tuned XGBoost params: {'learning_rate': 0.1, 'max_depth': 9, 'n_estimators': 500, 'subsample': 0.7}
Prior CV Macro F1: 0.8949


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [4]:
# Train default XGBoost for comparison
model_default = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=10,
    random_state=42,
    tree_method='hist',
)
model_default.fit(X_train, y_train)
y_pred_default = model_default.predict(X_test)
macro_f1_default = f1_score(y_test, y_pred_default, average='macro')

# Evaluate tuned model on test set
y_pred_tuned = model.predict(X_test)
macro_f1_tuned = f1_score(y_test, y_pred_tuned, average='macro')

print("=" * 70)
print("Default XGBoost")
print("=" * 70)
print(f"Macro F1: {macro_f1_default:.4f}\n")
print(classification_report(y_test, y_pred_default, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Tuned XGBoost {best_params}")
print("=" * 70)
print(f"Macro F1: {macro_f1_tuned:.4f}\n")
print(classification_report(y_test, y_pred_tuned, target_names=le_y.classes_))

print("\n" + "=" * 70)
print(f"Improvement: {macro_f1_default:.4f} -> {macro_f1_tuned:.4f} ({macro_f1_tuned - macro_f1_default:+.4f})")
print("=" * 70)

Default XGBoost
Macro F1: 0.8887

                          precision    recall  f1-score   support

                 Drought       1.00      1.00      1.00      1858
          Excessive Heat       0.93      0.82      0.87      1537
             Flash Flood       0.87      0.87      0.87      1657
                   Flood       0.88      0.80      0.84      1064
                    Hail       0.74      0.65      0.69      4140
                    Heat       0.81      0.92      0.86      1299
               High Wind       0.96      0.95      0.96      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       967
       Thunderstorm Wind       0.82      0.88      0.85      8235
          Winter Weather       0.95      0.96      0.95      1290

                accuracy                           0.86     23544
               macro avg       0.90      0.88      0.89     23544
            weighted avg       0.86      0.86      0.86     23544


Tuned XGBoost {'learning_rate': 0.1, '

In [5]:
# Confusion matrices side by side
cm_default = confusion_matrix(y_test, y_pred_default)
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

print("=" * 70)
print("Confusion Matrix — Default XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_default, index=le_y.classes_, columns=le_y.classes_))

print("\n" + "=" * 70)
print("Confusion Matrix — Tuned XGBoost")
print("=" * 70)
print(pd.DataFrame(cm_tuned, index=le_y.classes_, columns=le_y.classes_))

# Highlight Hail <-> Thunderstorm Wind confusion (dominant error pattern)
print("\n" + "=" * 70)
print("Hail <-> Thunderstorm Wind confusion")
print("=" * 70)
hail_idx = list(le_y.classes_).index('Hail')
tsw_idx = list(le_y.classes_).index('Thunderstorm Wind')
print(f"  Default — Hail predicted as TStorm Wind: {cm_default[hail_idx, tsw_idx]}")
print(f"  Tuned   — Hail predicted as TStorm Wind: {cm_tuned[hail_idx, tsw_idx]}")
print(f"  Default — TStorm Wind predicted as Hail: {cm_default[tsw_idx, hail_idx]}")
print(f"  Tuned   — TStorm Wind predicted as Hail: {cm_tuned[tsw_idx, hail_idx]}")

# Feature importance comparison
imp_default = pd.Series(model_default.feature_importances_, index=feature_cols)
imp_tuned = pd.Series(model.feature_importances_, index=feature_cols)
imp_cmp = pd.DataFrame({
    'Default': imp_default,
    'Tuned': imp_tuned,
    'Diff': imp_tuned - imp_default,
}).sort_values('Default', ascending=False)

print("\n" + "=" * 70)
print("Feature Importance Comparison")
print("=" * 70)
print(imp_cmp.to_string(float_format='%.4f'))

Confusion Matrix — Default XGBoost
                          Drought  Excessive Heat  Flash Flood  Flood  Hail  \
Drought                      1853               3            0      0     0   
Excessive Heat                  3            1258            0      0     0   
Flash Flood                     0               0         1437     93    27   
Flood                           0               0          168    847    10   
Hail                            0               0           11      6  2692   
Heat                            4              98            0      0     0   
High Wind                       0               1            0      1     0   
Marine Thunderstorm Wind        0               0            0      0     0   
Thunderstorm Wind               0               0           31     13   927   
Winter Weather                  0               0            0      0     0   

                          Heat  High Wind  Marine Thunderstorm Wind  \
Drought                 

# TabNet Baseline

In [6]:
import torch
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier

# Reuse the same X_train, X_test, y_train, y_test from the XGBoost cells above
# TabNet needs numpy float32 arrays
X_train_np = X_train.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

# Categorical feature indices and dimensions for TabNet embeddings
cat_cols_tabnet = ['STATE_ENC', 'CZ_TYPE_ENC', 'WFO_ENC']
cat_idxs = [feature_cols.index(c) for c in cat_cols_tabnet]
cat_dims = [int(df_imp[c].nunique()) for c in cat_cols_tabnet]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tabnet = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=1,
    n_d=8,
    n_a=8,
    n_steps=3,
    gamma=1.3,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device,
    verbose=10,
)

tabnet.fit(
    X_train_np, y_train,
    eval_set=[(X_test_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100,
    patience=15,
    batch_size=1024,
)

Using device: cuda


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.22854 | val_0_accuracy: 0.18994 |  0:00:01s
epoch 10 | loss: 0.50437 | val_0_accuracy: 0.39896 |  0:00:14s
epoch 20 | loss: 0.47449 | val_0_accuracy: 0.40414 |  0:00:27s

Early stopping occurred at epoch 22 with best_epoch = 7 and best_val_0_accuracy = 0.77931


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [7]:
# TabNet evaluation
y_pred_tabnet = tabnet.predict(X_test_np)
macro_f1_tabnet = f1_score(y_test, y_pred_tabnet, average='macro')

print("=" * 70)
print("TabNet Baseline")
print("=" * 70)
print(f"Macro F1: {macro_f1_tabnet:.4f}\n")
print(classification_report(y_test, y_pred_tabnet, target_names=le_y.classes_))

# Compare all three models
print("\n" + "=" * 70)
print("Model Comparison (Macro F1)")
print("=" * 70)
print(f"  Default XGBoost:  {macro_f1_default:.4f}")
print(f"  Tuned XGBoost:    {macro_f1_tuned:.4f}")
print(f"  TabNet Baseline:  {macro_f1_tabnet:.4f}")

# Confusion matrix
cm_tabnet = confusion_matrix(y_test, y_pred_tabnet)
print("\n" + "=" * 70)
print("Confusion Matrix — TabNet")
print("=" * 70)
print(pd.DataFrame(cm_tabnet, index=le_y.classes_, columns=le_y.classes_))

# TabNet feature importances
imp_tabnet = pd.Series(tabnet.feature_importances_, index=feature_cols)
print("\n" + "=" * 70)
print("TabNet Feature Importance")
print("=" * 70)
print(imp_tabnet.sort_values(ascending=False).to_string(float_format='%.4f'))

TabNet Baseline
Macro F1: 0.7828

                          precision    recall  f1-score   support

                 Drought       0.98      0.97      0.97      1858
          Excessive Heat       0.64      0.81      0.71      1537
             Flash Flood       0.78      0.84      0.81      1657
                   Flood       0.89      0.58      0.71      1064
                    Hail       0.67      0.53      0.59      4140
                    Heat       0.68      0.44      0.53      1299
               High Wind       0.87      0.81      0.84      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       967
       Thunderstorm Wind       0.77      0.87      0.82      8235
          Winter Weather       0.80      0.90      0.85      1290

                accuracy                           0.78     23544
               macro avg       0.81      0.78      0.78     23544
            weighted avg       0.78      0.78      0.77     23544


Model Comparison (Macro F1)
  Default 

# Ablation Study: CZ_TYPE Removed

Retrain all three models (default XGBoost, tuned XGBoost, TabNet) with `CZ_TYPE_ENC` dropped to measure its true contribution and verify the remaining meteorological features can carry performance.

In [8]:
# Ablation: drop CZ_TYPE_ENC and retrain each model
feature_cols_ablation = [c for c in feature_cols if c != 'CZ_TYPE_ENC']
X_train_abl = X_train[feature_cols_ablation]
X_test_abl = X_test[feature_cols_ablation]

# Default XGBoost (no CZ_TYPE)
model_default_abl = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
)
model_default_abl.fit(X_train_abl, y_train)
y_pred_default_abl = model_default_abl.predict(X_test_abl)
macro_f1_default_abl = f1_score(y_test, y_pred_default_abl, average='macro')

# Tuned XGBoost (no CZ_TYPE) — reuse best params from Week 2 grid search
model_tuned_abl = xgb.XGBClassifier(
    objective='multi:softmax', num_class=10, random_state=42, tree_method='hist',
    **best_params,
)
model_tuned_abl.fit(X_train_abl, y_train)
y_pred_tuned_abl = model_tuned_abl.predict(X_test_abl)
macro_f1_tuned_abl = f1_score(y_test, y_pred_tuned_abl, average='macro')

# TabNet (no CZ_TYPE) — rebuild cat indices/dims since CZ_TYPE_ENC is gone
X_train_abl_np = X_train_abl.values.astype(np.float32)
X_test_abl_np = X_test_abl.values.astype(np.float32)
cat_cols_abl = ['STATE_ENC', 'WFO_ENC']
cat_idxs_abl = [feature_cols_ablation.index(c) for c in cat_cols_abl]
cat_dims_abl = [int(df_imp[c].nunique()) for c in cat_cols_abl]

tabnet_abl = TabNetClassifier(
    cat_idxs=cat_idxs_abl, cat_dims=cat_dims_abl, cat_emb_dim=1,
    n_d=8, n_a=8, n_steps=3, gamma=1.3,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_abl.fit(
    X_train_abl_np, y_train,
    eval_set=[(X_test_abl_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
y_pred_tabnet_abl = tabnet_abl.predict(X_test_abl_np)
macro_f1_tabnet_abl = f1_score(y_test, y_pred_tabnet_abl, average='macro')

print("=" * 70)
print("Ablation Study — CZ_TYPE_ENC removed")
print("=" * 70)
print(f"{'Model':<25} {'With CZ_TYPE':>14} {'Without':>10} {'Delta':>10}")
print("-" * 70)
print(f"{'Default XGBoost':<25} {macro_f1_default:>14.4f} {macro_f1_default_abl:>10.4f} {macro_f1_default_abl - macro_f1_default:>+10.4f}")
print(f"{'Tuned XGBoost':<25} {macro_f1_tuned:>14.4f} {macro_f1_tuned_abl:>10.4f} {macro_f1_tuned_abl - macro_f1_tuned:>+10.4f}")
print(f"{'TabNet':<25} {macro_f1_tabnet:>14.4f} {macro_f1_tabnet_abl:>10.4f} {macro_f1_tabnet_abl - macro_f1_tabnet:>+10.4f}")

print("\n" + "=" * 70)
print("Tuned XGBoost (no CZ_TYPE) — per-class report")
print("=" * 70)
print(classification_report(y_test, y_pred_tuned_abl, target_names=le_y.classes_))

# Feature importance without CZ_TYPE — check what the model relies on instead
imp_tuned_abl = pd.Series(model_tuned_abl.feature_importances_, index=feature_cols_ablation)
print("=" * 70)
print("Tuned XGBoost Feature Importance (no CZ_TYPE)")
print("=" * 70)
print(imp_tuned_abl.sort_values(ascending=False).to_string(float_format='%.4f'))

/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.5312  | val_0_accuracy: 0.15048 |  0:00:01s
epoch 10 | loss: 0.63656 | val_0_accuracy: 0.37772 |  0:00:14s
epoch 20 | loss: 0.59575 | val_0_accuracy: 0.70761 |  0:00:27s
epoch 30 | loss: 0.58477 | val_0_accuracy: 0.70816 |  0:00:40s
epoch 40 | loss: 0.5707  | val_0_accuracy: 0.39169 |  0:00:53s
epoch 50 | loss: 0.56342 | val_0_accuracy: 0.69343 |  0:01:06s
epoch 60 | loss: 0.55343 | val_0_accuracy: 0.38231 |  0:01:18s
epoch 70 | loss: 0.54302 | val_0_accuracy: 0.76818 |  0:01:31s
epoch 80 | loss: 0.53954 | val_0_accuracy: 0.77727 |  0:01:45s

Early stopping occurred at epoch 82 with best_epoch = 67 and best_val_0_accuracy = 0.78071


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Ablation Study — CZ_TYPE_ENC removed
Model                       With CZ_TYPE    Without      Delta
----------------------------------------------------------------------
Default XGBoost                   0.8887     0.8738    -0.0149
Tuned XGBoost                     0.9001     0.8871    -0.0130
TabNet                            0.7828     0.7871    +0.0043

Tuned XGBoost (no CZ_TYPE) — per-class report
                          precision    recall  f1-score   support

                 Drought       1.00      1.00      1.00      1858
          Excessive Heat       0.88      0.85      0.86      1537
             Flash Flood       0.87      0.87      0.87      1657
                   Flood       0.84      0.77      0.80      1064
                    Hail       0.78      0.73      0.75      4140
                    Heat       0.83      0.87      0.85      1299
               High Wind       0.94      0.90      0.92      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       96

# TabNet Hyperparameter Tuning

Search over `n_d`/`n_a` width, `n_steps`, and learning rate. TabNet ties `n_d = n_a` by convention, so we sweep them together. Uses a single held-out validation split (carved from training) rather than full k-fold CV to keep runtime manageable.

In [9]:
import itertools

# Carve a validation set from training so the test set stays untouched during tuning
X_tr_tn, X_val_tn, y_tr_tn, y_val_tn = train_test_split(
    X_train_np, y_train, stratify=y_train, test_size=0.2, random_state=42
)

tabnet_grid = {
    'n_da': [8, 16, 32],         # n_d == n_a
    'n_steps': [3, 5, 7],
    'lr': [1e-2, 2e-2, 5e-2],
}

results = []
configs = list(itertools.product(tabnet_grid['n_da'], tabnet_grid['n_steps'], tabnet_grid['lr']))
print(f"Running {len(configs)} TabNet configs...")

for i, (n_da, n_steps, lr) in enumerate(configs, 1):
    print(f"\n[{i}/{len(configs)}] n_d=n_a={n_da}, n_steps={n_steps}, lr={lr}")
    clf = TabNetClassifier(
        cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
        n_d=n_da, n_a=n_da, n_steps=n_steps, gamma=1.3,
        optimizer_params=dict(lr=lr),
        scheduler_params={"step_size": 50, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        device_name=device, verbose=0,
    )
    clf.fit(
        X_tr_tn, y_tr_tn,
        eval_set=[(X_val_tn, y_val_tn)],
        eval_metric=["accuracy"],
        max_epochs=100, patience=15, batch_size=1024,
    )
    val_pred = clf.predict(X_val_tn)
    val_f1 = f1_score(y_val_tn, val_pred, average='macro')
    results.append({'n_da': n_da, 'n_steps': n_steps, 'lr': lr, 'val_macro_f1': val_f1})
    print(f"  val macro F1: {val_f1:.4f}")

results_df = pd.DataFrame(results).sort_values('val_macro_f1', ascending=False).reset_index(drop=True)
print("\n" + "=" * 70)
print("TabNet Tuning Results (sorted by val macro F1)")
print("=" * 70)
print(results_df.to_string(float_format='%.4f'))

best = results_df.iloc[0]
print(f"\nBest config: n_d=n_a={int(best['n_da'])}, n_steps={int(best['n_steps'])}, lr={best['lr']}")
print(f"Best val macro F1: {best['val_macro_f1']:.4f}")

# Retrain best config on full training set and evaluate on held-out test set
tabnet_best = TabNetClassifier(
    cat_idxs=cat_idxs, cat_dims=cat_dims, cat_emb_dim=1,
    n_d=int(best['n_da']), n_a=int(best['n_da']), n_steps=int(best['n_steps']), gamma=1.3,
    optimizer_params=dict(lr=float(best['lr'])),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    device_name=device, verbose=10,
)
tabnet_best.fit(
    X_train_np, y_train,
    eval_set=[(X_test_np, y_test)],
    eval_metric=["accuracy"],
    max_epochs=100, patience=15, batch_size=1024,
)
y_pred_tabnet_best = tabnet_best.predict(X_test_np)
macro_f1_tabnet_best = f1_score(y_test, y_pred_tabnet_best, average='macro')

print("\n" + "=" * 70)
print("Final Model Comparison (Macro F1 on test set)")
print("=" * 70)
print(f"  Default XGBoost:      {macro_f1_default:.4f}")
print(f"  Tuned XGBoost:        {macro_f1_tuned:.4f}")
print(f"  TabNet Baseline:      {macro_f1_tabnet:.4f}")
print(f"  TabNet Tuned:         {macro_f1_tabnet_best:.4f}")

print("\n" + classification_report(y_test, y_pred_tabnet_best, target_names=le_y.classes_))

Running 27 TabNet configs...

[1/27] n_d=n_a=8, n_steps=3, lr=0.01

Early stopping occurred at epoch 37 with best_epoch = 22 and best_val_0_accuracy = 0.79554


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  val macro F1: 0.8148

[2/27] n_d=n_a=8, n_steps=3, lr=0.02

Early stopping occurred at epoch 57 with best_epoch = 42 and best_val_0_accuracy = 0.8008


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  val macro F1: 0.8189

[3/27] n_d=n_a=8, n_steps=3, lr=0.05

Early stopping occurred at epoch 19 with best_epoch = 4 and best_val_0_accuracy = 0.76273


...

  val macro F1: 0.7422

TabNet Tuning Results (sorted by val macro F1)
    n_da  n_steps     lr  val_macro_f1
0     16        7 0.0200        0.8358
1     32        3 0.0200        0.8353
2     32        3 0.0100        0.8336
3     16        3 0.0200        0.8313
4     16        3 0.0100        0.8308
5      8        5 0.0100        0.8296
6     32        5 0.0100        0.8231
7      8        5 0.0500        0.8220
8     32        7 0.0200        0.8193
9      8        3 0.0200        0.8189
10     8        3 0.0100        0.8148
11    16        5 0.0100        0.8138
12    32        5 0.0500        0.8135
13    32        5 0.0200        0.8129
14     8        5 0.0200        0.8123
15     8        7 0.0200        0.8091
16    16        5 0.0200        0.8064
17    32        7 0.0100        0.8025
18    16        7 0.0100        0.8015
19    32        3 0.0500        0.7961
20    16        3 0.0500        0.7901
21    16        5 0.0500        0.7847
22     8        7 0.0100        

/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.25584 | val_0_accuracy: 0.16688 |  0:00:02s
epoch 10 | loss: 0.52537 | val_0_accuracy: 0.75565 |  0:00:27s
epoch 20 | loss: 0.49762 | val_0_accuracy: 0.47443 |  0:00:51s
epoch 30 | loss: 0.48226 | val_0_accuracy: 0.77689 |  0:01:16s
epoch 40 | loss: 0.47038 | val_0_accuracy: 0.78398 |  0:01:40s
epoch 50 | loss: 0.46206 | val_0_accuracy: 0.79286 |  0:02:05s
epoch 60 | loss: 0.46213 | val_0_accuracy: 0.76185 |  0:02:30s

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_0_accuracy = 0.80148


/home/nate/code/noaa-storm-classification/.venv/lib/python3.14/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Final Model Comparison (Macro F1 on test set)
  Default XGBoost:      0.8887
  Tuned XGBoost:        0.9001
  TabNet Baseline:      0.7828
  TabNet Tuned:         0.8231

                          precision    recall  f1-score   support

                 Drought       0.99      0.98      0.99      1858
          Excessive Heat       0.90      0.60      0.72      1537
             Flash Flood       0.79      0.85      0.82      1657
                   Flood       0.82      0.71      0.76      1064
                    Hail       0.67      0.59      0.63      4140
                    Heat       0.65      0.92      0.76      1299
               High Wind       0.89      0.85      0.87      1497
Marine Thunderstorm Wind       1.00      1.00      1.00       967
       Thunderstorm Wind       0.79      0.84      0.82      8235
          Winter Weather       0.85      0.89      0.87      1290

                accuracy                           0.80     23544
               macro avg       0.8